# 03 - BESS model validation (workflow step 3)

**Question:** does the DC-side BESS model (Eqs. bess_net_power, soc_dynamics, soc_bounds,
bess_power_bounds, usable_capacity) behave exactly as specified -- signs, efficiencies, limits,
energy conservation and the SOH state -- before any flexibility logic is added?

Every check is a pass/fail test with an explicit tolerance.
Output: `output/results/results_bess_validation.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd

from dfm_acdc.config import RESULTS_DIR, BESSParams, DegradationParams
from dfm_acdc.degradation import XuDegradation
from dfm_acdc.bess import BESSPlant, soc_feasible_bounds, split_power

b = BESSParams()
deg = XuDegradation(DegradationParams(), b.soc_max, b.soc_min)
dt = 0.25
tests = []
def check(name, value, ok, tol=""):
    tests.append({"test": name, "value": value, "tolerance": tol, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {value}")

## Sign convention and mutually exclusive charge/discharge

In [ ]:
pl = BESSPlant(b, deg)
r = pl.step(+60.0, dt); check("discharge (P_B>0) lowers SOC", r["soc_next"] - r["soc"], r["soc_next"] < r["soc"])
r = pl.step(-60.0, dt); check("charge (P_B<0) raises SOC", r["soc_next"] - r["soc"], r["soc_next"] > r["soc"])
ok = all(min(split_power(P)) == 0.0 and split_power(P)[0] - split_power(P)[1] == P for P in np.linspace(-150, 150, 61))
check("P_B = P_dis - P_ch with P_dis*P_ch = 0", "61 values", ok)

## SOC update and efficiencies (Eq. soc_dynamics)

In [ ]:
pl = BESSPlant(b, deg)
E_use = pl.E_use
r = pl.step(+100.0, dt)
exp = -100.0 / b.eta_dis * dt / E_use
check("discharge SOC step = -P dt/(eta_dis E_use)", r["soc_next"] - r["soc"] - exp, abs(r["soc_next"] - r["soc"] - exp) < 1e-12, "1e-12")
r = pl.step(-100.0, dt)
exp = b.eta_ch * 100.0 * dt / r["E_use"]          # usable energy at the start of the interval
check("charge SOC step = eta_ch P dt / E_use", r["soc_next"] - r["soc"] - exp, abs(r["soc_next"] - r["soc"] - exp) < 1e-12, "1e-12")
# round trip: charge then discharge back to the same SOC
pl = BESSPlant(b, deg); pl.reset(0.3)
E_in = 0.0
for _ in range(4):                                # 100 kWh in: stays inside the SOC window
    E_in += 100.0 * dt; pl.step(-100.0, dt)
soc_hi = pl.state.soc
E_out = 0.0
while pl.state.soc > 0.3 + 1e-12:
    P = min(100.0, (pl.state.soc - 0.3) * pl.E_use * b.eta_dis / dt)
    E_out += P * dt; pl.step(P, dt)
rt = E_out / E_in
check("round-trip efficiency = eta_ch*eta_dis", rt, abs(rt - b.eta_ch * b.eta_dis) < 1e-4, "1e-4 (SOH drift)")

## Energy conservation over a random 1000-interval dispatch

In [ ]:
rng = np.random.default_rng(7)
pl = BESSPlant(b, deg)
E_term = 0.0
for _ in range(1000):
    lo, hi = soc_feasible_bounds(pl.state.soc, pl.E_use, b, dt)
    P = rng.uniform(lo, hi)
    E_term += P * dt
    pl.step(P, dt)
L = pd.DataFrame(pl.log)
stored = np.sum((L.soc_next - L.soc) * L.E_use)
res = E_term + stored + L.E_loss.sum()
check("terminal energy + stored-energy change + losses = 0 [kWh]", res, abs(res) < 1e-9, "1e-9 kWh")
check("SOC never leaves [SOC_min, SOC_max]", f"[{L.soc_next.min():.4f}, {L.soc_next.max():.4f}]",
      L.soc_next.min() >= b.soc_min - 1e-12 and L.soc_next.max() <= b.soc_max + 1e-12)
check("no SOC clipping needed (bounds respected by construction)", L["clip"].abs().max(), L["clip"].abs().max() < 1e-12)
check("|P_B| <= P_rated", L.P_B.abs().max(), L.P_B.abs().max() <= b.P_rated_kw + 1e-12)

## Power and energy limits (Eq. bess_power_bounds with the SOC window)

In [ ]:
for soc in (0.1, 0.12, 0.5, 0.88, 0.9):
    lo, hi = soc_feasible_bounds(soc, 270.0, b, dt)
    s_lo = soc + b.eta_ch * (-lo) * dt / 270.0
    s_hi = soc - hi / b.eta_dis * dt / 270.0
    check(f"bounds at SOC={soc}: [{lo:.1f}, {hi:.1f}] kW keep SOC in window",
          f"{s_hi:.4f}..{s_lo:.4f}", s_hi >= b.soc_min - 1e-12 and s_lo <= b.soc_max + 1e-12)

## SOH state and usable energy (Eqs. degradation_state, soh, usable_capacity)

In [ ]:
check("SOH non-increasing over the random dispatch", float(np.diff(np.r_[b.soh_0, L.soh.values]).max()),
      np.all(np.diff(L.soh.values) <= 1e-15))
check("E_use = SOH * E_rated", float(np.max(np.abs(L.E_use - L.soh * b.E_rated_kwh))),
      np.max(np.abs(L.E_use - L.soh * b.E_rated_kwh)) < 1e-9)
pl = BESSPlant(b, deg)
for _ in range(96):
    pl.step(0.0, dt)
Lc = pd.DataFrame(pl.log)
check("idle day: zero cycle aging", Lc.dF_cyc.sum(), Lc.dF_cyc.sum() == 0.0)
cal = deg.dF_calendar(b.soc_0, 24.0) * deg.dL(deg.F_from_soh(b.soh_0))
check("idle day: SOH loss = calendar aging only", f"{b.soh_0 - pl.state.soh:.3e} vs {cal:.3e}",
      abs((b.soh_0 - pl.state.soh) - cal) / cal < 1e-3, "0.1 %")

In [ ]:
val = pd.DataFrame(tests)
val.to_csv(os.path.join(RESULTS_DIR, "results_bess_validation.csv"), index=False)
print(f"\n{val['pass'].sum()}/{len(val)} tests passed")
assert val["pass"].all()

## Takeaway

The BESS model has the stated sign convention, never charges and discharges simultaneously,
reproduces the charge/discharge efficiencies exactly, conserves energy to machine precision,
respects its power rating and SOC window by construction, and carries a monotone SOH state
that sets the usable energy $E^{use}_k = SOH_k E_{rated}$. An idle day ages the battery only
through calendar aging.